# GDELT 중동 국가 간 선택 EventCode 분석 (수정본)

## 분석 목적
1. `Actor1CountryCode`와 `Actor2CountryCode`가 **모두 지정한 중동 16개국**이고, **서로 다른 나라**인 이벤트를 필터링합니다.
2. 그중 지정한 20개 `EventCode`만 필터링합니다.
3. `Actor1 → Actor2` 방향을 유지하여 **국가쌍별 이벤트 수**를 집계합니다.

> `Actor1CountryCode`, `Actor2CountryCode`는 ISO 2자리 코드가 아니라 GDELT/CAMEO의 3자리 국가 코드를 사용합니다.
> `EventCode`는 `1381`처럼 계층형 코드이며 앞자리 0이 존재할 수 있으므로 문자열로 다루는 것이 안전합니다.

---

## ✏️ 수정 요약
코드에서 `✏️ [수정 N]` 으로 검색하면 바뀐 위치를 바로 찾을 수 있습니다.

| 번호 | 위치 | 무엇을 바꿨나 | 왜 |
|---|---|---|---|
| 수정 1 | 0. 설정 | 날짜·폴더를 변수 하나로 관리 | 파일 경로와 저장 파일명에 `20260913`이 곳곳에 박혀 있어 다른 날짜를 돌리기 어려웠음 |
| 수정 2 | 1. 불러오기 | 압축 파일(.zip)·서버 주소도 바로 읽기, 2013년 3월 이전 57열 파일 대응 | 서버 원본은 `.zip`이고, 옛날 파일은 `SOURCEURL` 열이 없어 58열 헤더를 쓰면 열이 밀림 |
| 수정 3 | 2. 16개국 필터 | 자국 내 이벤트(`ISR→ISR` 등) 제외를 **필터 안에서** 적용 | 원래 주석 코드는 `filtered`를 만든 **뒤에** 있어서 주석을 풀어도 결과가 바뀌지 않았음 |
| 수정 4 | 2. 16개국 필터 | `SQLDATE`(사건 날짜)가 너무 오래된 이벤트 제외 | 첫 행이 `20250913`처럼 1년 전 사건을 언급한 기사가 섞여 있었음 |
| 수정 5 | 3. EventCode | 코드 목록을 한 곳에서만 정의 (한국어 설명 포함) | 3번·4번 셀에 같은 목록과 필터가 두 번 있었음 |
| 수정 6 | 4. 컬럼 정리 | 숫자 열을 숫자형으로 변환 | `dtype=str`로 읽어서 `GoldsteinScale` 등으로 평균·합계를 낼 수 없었음 |
| 수정 7 | 6-2. QuadClass | 없는 QuadClass 열은 0으로 채움 | 어떤 날 QuadClass 3 사건이 하나도 없으면 `KeyError`가 났음 |
| 수정 8 | 5. 국가쌍별 이벤트 수 | `pair_counts` 표와 저장 파일 삭제 | 6-2 표의 `total_event_count`와 숫자가 완전히 같음 (20개 코드가 모두 QuadClass 3·4라서 3 + 4 = 전체) |
| 수정 9 | 0. 설정 · 1. 불러오기 | 결과를 `output/2026.8/20260831/`처럼 월 → 날짜 폴더에 저장, 월 폴더를 날짜에서 자동 계산 | 한 달치 결과 파일이 한 폴더에 섞이고, 월(`2026.8`)이 코드에 고정돼 9월로 바꾸기 번거로웠음 |
| 수정 13 | 8. 월 단위 처리 | 월간 결과 저장 위치를 `output/2026.8/월간/`으로 변경 | 날짜 폴더 31개와 월간 파일이 한 폴더에 섞이지 않도록 |
| 수정 12 | 0. 설정 | 원본 폴더 위치를 `event/data/2026.8_data`로 변경 | 월별 원본 폴더 162개를 `data` 폴더 하나에 모음 |
| 수정 11 | 0. 설정 | `EVENT_DIR = Path.cwd()` (노트북이 있는 폴더 기준) | 작성자 PC 경로가 고정돼 있어 팀원이 받으면 경로를 고쳐야 했음 |
| 수정 10 | 8. 월 단위 처리 | 한 달의 날짜별 결과(날짜 폴더) + 월간 표 3개와 일별 추이 표(월 폴더) 생성 | 하루치 표와 같은 형태의 월간 결과가 필요 |

## 0. 설정

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

# ✏️ [수정 1] 날짜와 폴더를 한 곳에서 관리합니다.
#   (원래) file_path = "data/20260913.export.CSV"  → 날짜가 경로·저장 파일명에 직접 들어가 있었음
DATE = "20260831"                 # 분석할 날짜 (YYYYMMDD). 여기만 바꾸면 전체가 바뀝니다.

# ✏️ [수정 11] 노트북이 있는 폴더(event 폴더)를 기준으로 동작합니다. 팀원 PC에서도 경로를 고칠 필요가 없습니다.
#   (원래) EVENT_DIR = Path(r"C:\Users\김관우\OneDrive\바탕 화면\event")  → 작성자 PC에서만 동작
#   ※ 노트북 파일은 반드시 data · output 폴더와 같은 위치(팀 저장소의 GDELT_folder)에 두고 여세요.
EVENT_DIR = Path.cwd()
if not ((EVENT_DIR / "output").exists() and (EVENT_DIR / "data").exists()):
    print("⚠️ 현재 폴더에 data · output 폴더가 없습니다. 노트북 파일이 있는 폴더(GDELT_folder)를 기준으로 실행 중인지 확인하세요:", EVENT_DIR)
    print("   처음이라면 download_gdelt_month.py로 원본을 먼저 받으세요. (폴더가 자동으로 만들어집니다)")


# ✏️ [수정 9] 날짜별 폴더 구조로 변경합니다.
#   event/data/2026.8_data/             ← 원본 zip (월별)   ※ [수정 12]에서 data 폴더 아래로 이동
#   event/output/2026.8/20260831/       ← 결과 CSV (월 → 날짜)
#   (원래) DATA_DIR = EVENT_DIR / "2026.8_data"
#          OUTPUT_DIR = EVENT_DIR / "output" / "2026.8"   → 한 달치 결과가 한 폴더에 섞였고, 월이 코드에 고정돼 있었음
def month_label(date):
    """'20260831' → '2026.8' (기존 폴더 이름 형식)"""
    return f"{date[:4]}.{int(date[4:6])}"


def data_dir_for(date):
    # ✏️ [수정 12] 원본 폴더를 event/data/ 아래로 모았습니다.
    #   (원래) return EVENT_DIR / f"{month_label(date)}_data"   → event/2026.8_data
    return EVENT_DIR / "data" / f"{month_label(date)}_data"   # → event/data/2026.8_data


def output_dir_for(date):
    return EVENT_DIR / "output" / month_label(date) / date


DATA_DIR = data_dir_for(DATE)                 # 예: event/data/2026.8_data
OUTPUT_DIR = output_dir_for(DATE)             # 예: event/output/2026.8/20260831
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # parents=True: 상위 폴더가 없어도 함께 생성

print("원본 폴더:", DATA_DIR)
print("결과 폴더:", OUTPUT_DIR)

# 로컬에 파일이 없으면 GDELT 서버에서 바로 읽을지 여부
USE_SERVER_IF_MISSING = True
GDELT_URL = "https://data.gdeltproject.org/events/"

## 1. GDELT 1.0 Daily Event 파일 불러오기

In [ ]:
headers = [
    "GLOBALEVENTID","SQLDATE","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode","Actor1EthnicCode",
    "Actor1Religion1Code","Actor1Religion2Code","Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode","Actor2EthnicCode",
    "Actor2Religion1Code","Actor2Religion2Code","Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass","GoldsteinScale",
    "NumMentions","NumSources","NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode","Actor1Geo_ADM1Code",
    "Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode","Actor2Geo_ADM1Code",
    "Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode","ActionGeo_ADM1Code",
    "ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL"
]


def find_source(date):
    """✏️ [수정 2] 로컬 CSV → 로컬 ZIP → 서버 주소 순서로 읽을 파일을 찾습니다.
    (원래) 압축을 푼 로컬 CSV 한 가지만 읽을 수 있었음
    """
    folder = data_dir_for(date)   # ✏️ [수정 9] 날짜에 맞는 월 폴더에서 찾음 (원래: DATA_DIR 고정)
    for name in [f"{date}.export.CSV", f"{date}.export.CSV.zip"]:
        if (folder / name).exists():
            return folder / name
    if USE_SERVER_IF_MISSING:
        return GDELT_URL + f"{date}.export.CSV.zip"
    raise FileNotFoundError(f"{folder} 안에 {date}.export.CSV 파일이 없습니다.")


def load_gdelt(date):
    source = find_source(date)
    # ✏️ [수정 2] 2013년 3월 31일 이전 파일은 SOURCEURL 열이 없는 57열 형식입니다.
    #   58열 헤더를 그대로 쓰면 열이 하나씩 밀리므로 날짜로 구분합니다.
    #   (참고: 2013.3 이전 원본은 일 단위가 아니라 1980.zip(연), 200601.zip(월) 형태로 제공됩니다.)
    names = headers if date >= "20130401" else headers[:-1]
    data = pd.read_csv(
        source,
        sep="\t",
        header=None,
        names=names,
        dtype=str,
        compression="infer",   # .zip이면 자동으로 압축을 풀어서 읽음
    )
    print(f"불러온 파일: {source}")
    return data


df = load_gdelt(DATE)

print("원본 데이터 크기:", df.shape)
df.head()

## 2. 중동 16개국 정의 및 Actor1·Actor2 필터링

In [ ]:
middle_east_cameo = {
    "SAU": "사우디아라비아",
    "UAE": "아랍에미리트",
    "QAT": "카타르",
    "KUW": "쿠웨이트",
    "BHR": "바레인",
    "OMA": "오만",
    "YEM": "예멘",
    "SYR": "시리아",
    "LBN": "레바논",
    "JOR": "요르단",
    "ISR": "이스라엘",
    "PSE": "팔레스타인",
    "IRQ": "이라크",
    "IRN": "이란",
    "EGY": "이집트",
    "TUR": "튀르키예"
}

both_middle_east = (
    df["Actor1CountryCode"].isin(middle_east_cameo) &
    df["Actor2CountryCode"].isin(middle_east_cameo)
)

# ✏️ [수정 3] 자국 내 이벤트(ISR→ISR, IRN→IRN 등)를 필터 단계에서 바로 제외합니다.
#   (원래) 4번 셀에 주석으로만 있었고, filtered를 만든 뒤에 위치해서 주석을 풀어도 효과가 없었음
different_country = df["Actor1CountryCode"] != df["Actor2CountryCode"]

# ✏️ [수정 4] 사건 날짜(SQLDATE)가 분석 날짜보다 너무 오래된 이벤트는 제외합니다.
#   GDELT는 "그날 수집한 기사"를 모으므로, 기사가 과거 사건을 언급하면 옛날 날짜가 섞입니다.
#   (원래) 날짜 필터가 없어 첫 행에 1년 전 날짜(20250913)가 들어 있었음
MAX_LAG_DAYS = 7   # 분석 날짜 기준 며칠 전 사건까지 인정할지
min_sqldate = (datetime.strptime(DATE, "%Y%m%d") - timedelta(days=MAX_LAG_DAYS)).strftime("%Y%m%d")
recent = df["SQLDATE"].between(min_sqldate, DATE)

print("16개국 간 이벤트 (필터 전)       :", both_middle_east.sum())
print(" - 자국 내 이벤트로 제외         :", (both_middle_east & ~different_country).sum())
print(" - 오래된 사건 날짜로 제외        :", (both_middle_east & different_country & ~recent).sum())

middle_east_events = df[both_middle_east & different_country & recent].copy()

print("최종: 서로 다른 중동 국가 간 최근 이벤트 수:", len(middle_east_events))
middle_east_events.head()

## 3. 지정 EventCode 20개 필터링

In [ ]:
# ✏️ [수정 5] EventCode 목록을 여기 한 곳에서만 정의합니다. (한국어 설명을 함께 넣음)
#   (원래) 3번 셀과 4번 셀에 같은 목록과 같은 필터 코드가 두 번 있었음
event_code_labels = {
    "138": "군사력 사용 위협",
    "1381": "봉쇄 위협",
    "1382": "점령 위협",
    "1383": "비정규 폭력 사용 위협",
    "1384": "재래식 군사 공격 위협",
    "1385": "대량살상무기 공격 위협",
    "139": "최후통첩",

    "152": "군 경계태세 강화",
    "154": "군 병력 동원·증강",

    "163": "금수·보이콧·제재 부과",

    "190": "재래식 군사력 사용",
    "191": "봉쇄·이동 제한",
    "192": "영토 점령",
    "193": "소형무기·경무기를 이용한 전투",
    "194": "포병·전차를 이용한 전투",
    "195": "공중무기 사용",
    "196": "휴전 위반",

    "204": "대량살상무기 사용",
    "2041": "화학·생물학·방사능 무기 사용",
    "2042": "핵무기 폭발",
}
target_event_codes = list(event_code_labels)

filtered = middle_east_events[
    middle_east_events["EventCode"].isin(target_event_codes)
].copy()

print("선택 EventCode에 해당하는 최종 이벤트 수:", len(filtered))
(
    filtered["EventCode"].value_counts().sort_index()
    .rename("event_count").to_frame()
    .assign(설명=lambda t: t.index.map(event_code_labels))
)

## 4. 국가명 추가 및 분석 컬럼 정리

- 컬럼 중 필요한 컬럼
    - (핵심적으로 필요하다고 생각해서 추린 것으로 추후 얼마든지 수정 가능합니다)
```
    GLOBALEVENTID         이벤트 고유식별번호
    SQLDATE               이벤트 발생날짜

    Actor1Name            행동 주체의 이름
    Actor1CountryCode     actor1이 속한 국가의 CAMEO 국가코드

    Actor2Name            actor1의 행동이 향한 대상의 이름
    Actor2CountryCode     actor2가 속한 국가의 CAMEO 국가코드

    EventCode             CAMEO 세부 사건 코드
    EventRootCode         EventCode의 상위 대분류코드
    QuadClass             CAMEO 사건을 4개로 분류 (1 언어적 협력 / 2 물질적 협력 / 3 언어적 갈등 / 4 물질적 갈등)

    GoldsteinScale        해당 사건 유형의 협력/갈등 정도를 나타내는 goldstein 점수
    NumMentions           해당 이벤트가 GDELT 데이터에서 언급된 횟수(사건의 언론 노출, 주목도 참고)
    NumSources            해당 이벤트를 보도한 서로 다른 뉴스 출처 수
    NumArticles           해당 이벤트와 연결된 기사 수
    AvgTone               관련 기사들의 평균적인 보도 논조(긍정, 부정)

    ActionGeo_FullName    해당 행동이 발생한 장소 이름
    ActionGeo_CountryCode 행동 발생 장소가 속한 국가코드
    SOURCEURL             해당 이벤트와 연결된 원문 뉴스 기사 URL
```

In [ ]:
filtered["Actor1Country_KR"] = filtered["Actor1CountryCode"].map(middle_east_cameo)
filtered["Actor2Country_KR"] = filtered["Actor2CountryCode"].map(middle_east_cameo)
filtered["EventCode_KR"] = filtered["EventCode"].map(event_code_labels)   # ✏️ [수정 5] 설명 열 추가

# ✏️ [수정 6] 숫자 열을 숫자형으로 바꿉니다.
#   (원래) dtype=str로 읽은 상태 그대로라 "-10.0" 같은 문자열이어서 평균·합계 계산이 불가능했음
numeric_cols = ["GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone"]
filtered[numeric_cols] = filtered[numeric_cols].apply(pd.to_numeric, errors="coerce")

# 분석에 사용할 컬럼
show_cols = [
    "GLOBALEVENTID",
    "SQLDATE",

    "Actor1Name",
    "Actor1CountryCode",

    "Actor2Name",
    "Actor2CountryCode",

    "EventCode",
    "EventCode_KR",          # ✏️ [수정 5] 추가
    "EventRootCode",
    "QuadClass",

    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",

    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "SOURCEURL"
]
show_cols = [c for c in show_cols if c in filtered.columns]   # ✏️ [수정 2] 57열 파일에는 SOURCEURL이 없음

print(filtered[numeric_cols].dtypes)
filtered[show_cols].head(10)

## 5. 국가쌍별 이벤트 수

여기서는 GDELT의 방향성을 유지합니다.

- `Actor1` = 행동 주체
- `Actor2` = 행동 대상

따라서 `ISR → IRN`과 `IRN → ISR`은 서로 다른 국가쌍으로 집계합니다.

+ 해당 국가쌍은 Actor1CountryCode + Actor2CountryCode를 기준으로 한 단순한 기준입니다.
+ ✏️ [수정 3] 덕분에 이제 `ISR → ISR` 같은 자국 내 쌍은 나오지 않습니다.

> ✏️ **[수정 8] 이 단계의 `pair_counts` 표를 삭제했습니다.**
> (원래) `groupby(["Actor1CountryCode", "Actor2CountryCode"]).size()`로 국가쌍별 전체 건수를 따로 만들었음
> → 선택한 20개 코드는 모두 QuadClass 3 또는 4이므로, **6-2 표의 `total_event_count`(= 3 건수 + 4 건수)와 숫자가 완전히 같습니다.**
> 국가쌍별 전체 건수는 6-2 표에서 확인하세요. (2026-08-31 데이터 38개 국가쌍으로 일치 확인)

## 6. 국가쌍 × EventCode별 이벤트 수 확인

### 6-1. 단순 국가쌍 확인

In [ ]:
pair_event_counts = (
    filtered
    .groupby([
        "Actor1CountryCode", "Actor1Country_KR",
        "Actor2CountryCode", "Actor2Country_KR",
        "EventCode", "EventCode_KR"        # ✏️ [수정 5] 설명 열 함께 표시
    ])
    .size()
    .reset_index(name="event_count")
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

pair_event_counts.head(50)

### 6-2. 국가쌍별 QuadClass 3 / 4 이벤트 수 집계

In [ ]:
pair_quad_counts = (
    filtered
    .groupby([
        "Actor1CountryCode", "Actor1Country_KR",
        "Actor2CountryCode", "Actor2Country_KR",
        "QuadClass"
    ])
    .size()
    .unstack(fill_value=0)
    # ✏️ [수정 7] 그날 QuadClass 3 또는 4가 하나도 없으면 열 자체가 생기지 않아 KeyError가 납니다.
    #   없는 열은 0으로 만들어 둡니다.  (원래) 이 줄이 없었음
    .reindex(columns=["3", "4"], fill_value=0)
    .reset_index()
)
pair_quad_counts.columns.name = None

# 컬럼명 변경
pair_quad_counts = pair_quad_counts.rename(columns={
    "3": "QuadClass_3_count",
    "4": "QuadClass_4_count"
})

# 전체 이벤트 수도 추가
pair_quad_counts["total_event_count"] = (
    pair_quad_counts["QuadClass_3_count"] +
    pair_quad_counts["QuadClass_4_count"]
)

# 전체 이벤트 수가 많은 국가쌍부터 정렬
pair_quad_counts = (
    pair_quad_counts
    .sort_values("total_event_count", ascending=False)
    .reset_index(drop=True)
)

pair_quad_counts.head(30)

## 7. 분석용 CSV 저장

In [ ]:
# ✏️ [수정 1] 파일명에 DATE 변수를 사용하고 OUTPUT_DIR에 저장합니다.
#   (원래) "GDELT_20260913_중동_..." 처럼 날짜가 직접 쓰여 있어 날짜를 바꿔도 같은 이름으로 덮어썼음
filtered[show_cols].to_csv(
    OUTPUT_DIR / f"GDELT_{DATE}_중동_선택EventCode_필터링.csv",
    index=False,
    encoding="utf-8-sig"
)

# ✏️ [수정 8] pair_counts 저장 삭제 (6-2 QuadClass 표의 total_event_count와 같은 내용)
#   (원래) pair_counts.to_csv(OUTPUT_DIR / f"GDELT_{DATE}_중동_국가쌍별_이벤트수.csv", ...)

pair_event_counts.to_csv(
    OUTPUT_DIR / f"GDELT_{DATE}_중동_국가쌍_EventCode별_이벤트수.csv",
    index=False,
    encoding="utf-8-sig"
)

pair_quad_counts.to_csv(                                   # ✏️ [수정 1] 6-2 결과도 저장 (원래 저장 안 했음)
    OUTPUT_DIR / f"GDELT_{DATE}_중동_국가쌍별_QuadClass.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", OUTPUT_DIR.resolve())

## 8. 월 단위 처리 (날짜별 결과 + 월간 합산)

✏️ **[수정 10]** 기존 "여러 날짜 처리" 셀을 월 단위 처리로 바꿨습니다.
(원래) 날짜 범위를 받아 필터링 결과와 "일별 국가쌍 건수" 하나만 저장했음 → 하루치 표(6-1, 6-2)와 같은 형태의 **월간 표**가 없었음

`DATE`가 속한 달의 모든 날짜를 처리합니다. (`DATE = "20260831"`이면 2026년 8월 1일~31일)

| 저장 위치 | 파일 | 내용 |
|---|---|---|
| `output/2026.8/20260801/` … | `GDELT_20260801_중동_…csv` 3개 | 날짜별 결과 (위 1~7단계와 같은 표) |
| `output/2026.8/월간/` | `GDELT_2026.8_월간_중동_선택EventCode_필터링.csv` | 한 달치 이벤트 전체 |
| `output/2026.8/월간/` | `GDELT_2026.8_월간_중동_국가쌍_EventCode별_이벤트수.csv` | 월간 6-1 표 |
| `output/2026.8/월간/` | `GDELT_2026.8_월간_중동_국가쌍별_QuadClass.csv` | 월간 6-2 표 |
| `output/2026.8/월간/` | `GDELT_2026.8_월간_중동_일별_국가쌍_이벤트수.csv` | 국가쌍 × 날짜 건수 (추이용) |

✏️ [수정 13] 월간 결과는 `월간` 폴더에 저장됩니다. (원래: 월 폴더 바로 아래)

- `RUN_MONTH = True`로 바꿔야 실행됩니다. 한 달치 원본을 모두 읽으므로 몇 분 걸립니다.
- 월 폴더(`2026.8_data`)에 없는 날짜는 서버에서 읽습니다 (`USE_SERVER_IF_MISSING`).

In [ ]:
RUN_MONTH = False

# ✏️ [수정 10] 처리할 달은 DATE에서 자동으로 정합니다.
month_start = pd.Timestamp(DATE).replace(day=1)
MONTH_DAYS = pd.date_range(month_start, month_start + pd.offsets.MonthEnd(0)).strftime("%Y%m%d")
# ✏️ [수정 13] 월간 결과를 날짜 폴더와 섞이지 않게 '월간' 폴더에 저장합니다.
#   (원래) MONTH_DIR = EVENT_DIR / "output" / month_label(DATE)   → event/output/2026.8
MONTH_DIR = EVENT_DIR / "output" / month_label(DATE) / "월간"   # 예: event/output/2026.8/월간


def filter_middle_east(raw, date):
    """2~3단계(16개국 · 자국 제외 · 날짜 · EventCode 필터)를 한 번에 적용합니다."""
    min_date = (datetime.strptime(date, "%Y%m%d") - timedelta(days=MAX_LAG_DAYS)).strftime("%Y%m%d")
    mask = (
        raw["Actor1CountryCode"].isin(middle_east_cameo)
        & raw["Actor2CountryCode"].isin(middle_east_cameo)
        & (raw["Actor1CountryCode"] != raw["Actor2CountryCode"])
        & raw["SQLDATE"].between(min_date, date)
        & raw["EventCode"].isin(target_event_codes)
    )
    out = raw[mask].copy()
    out.insert(0, "DATE", date)
    return out


def add_columns(events):
    """4단계: 한국어 국가명 · 사건 설명 추가, 숫자 열 변환"""
    events["Actor1Country_KR"] = events["Actor1CountryCode"].map(middle_east_cameo)
    events["Actor2Country_KR"] = events["Actor2CountryCode"].map(middle_east_cameo)
    events["EventCode_KR"] = events["EventCode"].map(event_code_labels)
    events[numeric_cols] = events[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return events


def make_tables(events):
    """6-1(국가쌍 × EventCode), 6-2(국가쌍 × QuadClass) 표를 만듭니다. 하루치 · 월간 모두 같은 함수 사용."""
    pair_keys = ["Actor1CountryCode", "Actor1Country_KR", "Actor2CountryCode", "Actor2Country_KR"]
    pe = (
        events.groupby(pair_keys + ["EventCode", "EventCode_KR"]).size()
        .reset_index(name="event_count")
        .sort_values("event_count", ascending=False).reset_index(drop=True)
    )
    pq = (
        events.groupby(pair_keys + ["QuadClass"]).size()
        .unstack(fill_value=0).reindex(columns=["3", "4"], fill_value=0)
        .reset_index()
        .rename(columns={"3": "QuadClass_3_count", "4": "QuadClass_4_count"})
    )
    pq.columns.name = None
    pq["total_event_count"] = pq["QuadClass_3_count"] + pq["QuadClass_4_count"]
    pq = pq.sort_values("total_event_count", ascending=False).reset_index(drop=True)
    return pe, pq


def save_tables(folder, label, events, pe, pq):
    folder.mkdir(parents=True, exist_ok=True)
    cols = [c for c in ["DATE"] + show_cols if c in events.columns]
    events[cols].to_csv(folder / f"GDELT_{label}_중동_선택EventCode_필터링.csv", index=False, encoding="utf-8-sig")
    pe.to_csv(folder / f"GDELT_{label}_중동_국가쌍_EventCode별_이벤트수.csv", index=False, encoding="utf-8-sig")
    pq.to_csv(folder / f"GDELT_{label}_중동_국가쌍별_QuadClass.csv", index=False, encoding="utf-8-sig")


if RUN_MONTH:
    month_events = []
    for day in MONTH_DAYS:
        try:
            raw = load_gdelt(day)
        except Exception as e:          # 파일이 없거나 네트워크 오류가 나도 다음 날짜로 진행
            print(f"{day} 건너뜀: {e}")
            continue
        events = add_columns(filter_middle_east(raw, day))
        del raw                          # 원본(하루 약 10만 행)은 바로 메모리에서 지움
        pe, pq = make_tables(events)
        save_tables(output_dir_for(day), day, events, pe, pq)          # 날짜 폴더
        month_events.append(events)
        print(f"  {day}: {len(events)}건 → {output_dir_for(day).name}/")

    monthly = pd.concat(month_events, ignore_index=True)
    # 같은 사건이 두 날짜 파일에 겹쳐 들어가는 경우를 대비해 GLOBALEVENTID로 중복 제거
    before = len(monthly)
    monthly = monthly.drop_duplicates("GLOBALEVENTID")
    print(f"\n월간 이벤트: {len(monthly)}건 (중복 제거 {before - len(monthly)}건)")

    month_pe, month_pq = make_tables(monthly)
    label = f"{month_label(DATE)}_월간"
    save_tables(MONTH_DIR, label, monthly, month_pe, month_pq)                # 월 폴더

    daily_pair_counts = (
        monthly.groupby(["DATE", "Actor1CountryCode", "Actor1Country_KR", "Actor2CountryCode", "Actor2Country_KR"])
        .size().reset_index(name="event_count")
    )
    daily_pair_counts.to_csv(MONTH_DIR / f"GDELT_{label}_중동_일별_국가쌍_이벤트수.csv",
                             index=False, encoding="utf-8-sig")

    print("저장 완료:", MONTH_DIR)
    display(month_pq.head(15))